#Lead Data Engineer Task 
#PEI Group

**Scenario: E-commerce order Data Processing with Databricks
You've been assigned to design and implement a data processing system using Databricks for an e-commerce platform. This platform generates a lot of order data, including details about orders, products, customers, and transactions. Your goal is to use Databricks to create a scalable, efficient, and reliable data engineering solution. This solution should process and analyze the data to provide valuable insights for business stakeholders**

In [0]:
%python
%pip install openpyxl

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
from pyspark.sql.functions import *
import re

json_path = "/Volumes/workspace/default/pei/Orders.json"

df_orders = spark.read.format("json").option("multiline","true").load(json_path)

# Delta tables disallow ' ,;{}()\n\t= in column names, so sanitize them (e.g. "Customer ID" -> "Customer_ID")
df_orders = df_orders.toDF(*[re.sub(r"[ ,;{}()\n\t=]", "_", c) for c in df_orders.columns])

#display(df_orders)

df_orders.write.format("delta").mode("overwrite").saveAsTable("filter_order")

In [0]:
import pandas as pd
import re

excel_path = "/Volumes/workspace/default/pei/Customer.xlsx"

# Read Excel file
%pip install openpyxl
pdf = pd.read_excel(excel_path, engine="openpyxl").astype({'phone': 'str'})

# Convert Pandas DataFrame to Spark DataFrame
df_customer = spark.createDataFrame(pdf)

# Delta tables disallow ' ,;{}()\n\t= in column names, so sanitize them (e.g. "Customer ID" -> "Customer_ID")
df_customer = df_customer.toDF(*[re.sub(r"[ ,;{}()\n\t=]", "_", c) for c in df_customer.columns])

#display(df_customer)

#Create Raw Delta table
df_customer.write.format("delta").mode("overwrite").saveAsTable("filter_customer")

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%sql
drop table if exists filter_product ;

In [0]:
csv_path = "/Volumes/workspace/default/pei/Products.csv"

import re

df_products = spark.read.format("csv").option("header", "true").option("inferSchema", "true") .option("delimiter", ",").load(csv_path)

# Delta tables disallow ' ,;{}()\n\t= in column names, so sanitize them (e.g. "Product ID" -> "Product_ID")
df_products = df_products.toDF(*[re.sub(r"[ ,;{}()\n\t=]", "_", c) for c in df_products.columns])

#display(df_products)

df_products.write.format("delta").mode("overwrite").saveAsTable("filter_product")

In [0]:
df_prod_cust = df_customer.join(df_products,on="State",how="inner")

df_prod_cust.write.format("delta").mode("overwrite").saveAsTable("enriched_prod_cust")

#display(df_prod_cust)

In [0]:
from pyspark.sql.functions import col, round

# Rebuild df_enriched with correct join keys and column names
# (orders -> customers on Customer_ID, -> products on Product_ID)
df_enriched = (
    df_orders
    .join(df_customer, "Customer_ID", "inner")
    .join(df_products, "Product_ID", "inner")
    .select(
        col("Order_ID"),
        col("Order_Date"),
        round(col("Profit"), 2).alias("Profit"),
        col("Customer_Name"),
        col("Country"),
        col("Category"),
        col("Sub-Category")
    )
)

df_enriched.write.format("delta").mode("overwrite").saveAsTable("final_enriched_order")

In [0]:
from pyspark.sql.functions import year, sum, col, to_date

df_aggregate = (
    df_enriched
    .withColumn("Year", year(to_date(col("Order_Date"), "d/M/yyyy")))
    .groupBy(
        "Year",
        "Category",
        "Sub-Category",
        "Customer_Name"
    )
    .agg(
        round(sum("Profit"), 2).alias("Total_Profit")
    )
)

df_aggregate.write.format("delta").mode("overwrite").saveAsTable("agg_profit")

# **PEI Lead Data Engineer Task — Data Quality & Pipeline Tests**

In [0]:
# Variable and path value 
CUSTOMERS_TABLE = "workspace.default.filter_customer"
PRODUCTS_TABLE  = "workspace.default.filter_product"
ORDER_TABLE     = "workspace.default.filter_order"
ENRICHED_TABLE  = "workspace.default.final_enriched_order"
ENRICHED_CUSTOMER_PRODUCT_TABLE = "workspace.default.enriched_prod_cust"
CUSTOMERS_FILE = "/Volumes/workspace/default/pei/customers.json"
PRODUCTS_FILE  = "/Volumes/workspace/default/pei/products.xlsx"
order_FILE     = "/Volumes/workspace/default/pei/order.csv"

# Set True/False if source files are not accessible. It will skip or by-pass that test case
CHECK_SOURCE_FILES = True

# Expected source/enriched column names
CUSTOMER_ID = "Customer_ID"
CUSTOMER_NAME = "Customer_Name"
COUNTRY = "Country"
PRODUCT_ID = "Product_ID"
PRODUCT_NAME = "Product_Name"
CATEGORY = "Category"
SUBCATEGORY = "Sub-Category"
ORDER_ID = "Order_ID"
ORDER_DATE = "Order_Date"
PROFIT_COL = "Profit"

In [0]:
print('========== SOURCE LOADING START ==========')
print('[INFO] Reading Orders JSON:', json_path)
orders = spark.read.option("multiLine", True).json(json_path)

print('[INFO] Reading Customer Excel:', excel_path)
customers_pd = pd.read_excel(excel_path).astype({'phone': 'str'})
customers = spark.createDataFrame(customers_pd)

print('[INFO] Reading Products CSV:', csv_path)
products = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)

========== SOURCE LOADING START ==========
[INFO] Reading Orders JSON: /Volumes/workspace/default/pei/Orders.json
[INFO] Reading Customer Excel: /Volumes/workspace/default/pei/Customer.xlsx
[INFO] Reading Products CSV: /Volumes/workspace/default/pei/Products.csv


**Basic/Sanity check TC-01 to TC-05**

In [0]:

from pyspark.sql.functions import year, sum, col, to_date

# TC-01 to TC-05 1: No null check on OrderId
assert df_enriched.filter(
    col("Order_ID").isNull()
).count() == 0

print("no null check for order_id done")

# Test-case 2: Profit should not be null
assert df_enriched.filter(
    col("Profit").isNull()
).count() == 0

print("not null check done")

# Test-case 3: Profit should have maximum 2 decimal places
assert df_enriched.filter(
    col("Profit") != round(col("Profit"), 2)
).count() == 0

print("profit check decimal upto 2 digits is done")

# Test-case 4: Aggregate profit should match enriched order profit
total_enriched = df_enriched.agg(
    sum("Profit")
).first()[0]

total_aggregate = df_aggregate.agg(
    sum("Total_Profit")
).first()[0]

print("total aggregation is done")

# Test-case 5: schema  check - if all columns exists
required_columns = [
        "Order_ID",
        "Order_Date",
        "Customer_Name",
        "Country",
        "Category",
        "Sub-Category"
        ]
columns_check = df_enriched

for c in required_columns:
    assert c in required_columns, f"Missing column: {c}"

print("schema check done")

print("All test cases passed!")

no null check for order_id done
not null check done
profit check decimal upto 2 digits is done
total aggregation is done
schema check done
All test cases passed!


In [0]:
results = []
print("=" * 80)
print("[INFO] INITIALIZING TEST FRAMEWORK")
print("=" * 80)

## Test-case (TC-06)
def record_test(test_id, description, fn, expected_failure=False):
    print(f"\n{'=' * 80}")
    print(f"[START] {test_id} | {description}")
    print(f"{'=' * 80}")

    try:
        detail = fn()
        status = "PASS"

        if detail is None:
            detail = "Condition satisfied"

        print(f"[PASS] {test_id}")
        print(f"[DETAILS] {detail}")

    except Exception as exc:
        status = "FAIL"
        detail = str(exc)[:500]

        print(f"[FAIL] {test_id}")
        print(f"[ERROR] {detail}")

    results.append({
        "Test_ID": test_id,
        "Test": description,
        "Status": status,
        "Details": detail
    })

    print(f"[END] {test_id} | Status: {status}")
    print("=" * 80)

## Test-case (TC-07)
def require(condition, message):
    if not condition:
        raise AssertionError(message)

## Test-case (TC-08)
def table_df(table_name):
    print(f"[INFO] Fetching table: {table_name}")
    df = spark.table(table_name)
    print(f"[INFO] Table loaded successfully: {table_name}")
    return df

## Test-case (TC-09)
def has_table(table_name):
    exists = spark.catalog.tableExists(table_name)
    print(f"[INFO] Table existence check | Table: {table_name} | Exists: {exists}")
    return exists

## Test-case (TC-10)
def col_exists(df, col_name):
    exists = col_name in df.columns
    print(f"[INFO] Column existence check | Column: {col_name} | Exists: {exists}")
    return exists

## Test-case (TC-11)
def require_columns(df, columns, label):
    print(f"[INFO] Validating columns for: {label}")
    print(f"[INFO] Expected columns: {columns}")

    missing = [c for c in columns if c not in df.columns]

    require(not missing, f"{label}: missing column(s): {missing}")

    print(f"[PASS] All required columns exist for {label}")

## Test-case (TC-12)
def null_count(df, col_name):
    print(f"[INFO] Checking NULL values in column: {col_name}")

    count = df.filter(F.col(col_name).isNull()).count()

    print(f"[INFO] NULL count | Column: {col_name} | Count: {count}")
    return count

## Test-case (TC-13)
def blank_count(df, col_name):
    print(f"[INFO] Checking blank values in column: {col_name}")

    count = df.filter(
        F.col(col_name).isNotNull() &
        (F.trim(F.col(col_name).cast("string")) == "")
    ).count()

    print(f"[INFO] Blank count | Column: {col_name} | Count: {count}")
    return count

## Test-case (TC-14)
def duplicate_key_count(df, key):
    print(f"[INFO] Checking duplicate keys: {key}")

    count = (
        df.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(f"[INFO] Duplicate key check | Key: {key} | Count: {count}")
    return count


def run_test(test_id, description, condition, details=""):
    print(f"[INFO] Preparing assertion for {test_id}")

    def test_function():
        print(f"[INFO] Evaluating condition for {test_id}")

        # Evaluate the condition inside the function
        evaluated_condition = condition() if callable(condition) else condition

        require(
            evaluated_condition,
            details if details else "Assertion failed"
        )

        return details if details else "Condition satisfied"

    record_test(
        test_id,
        description,
        test_function
    )


print("[INFO] All test utility functions initialized successfully.")
print(f"[INFO] Results list initialized. Current test count: {len(results)}")
print("[INFO] Ready to execute test cases.")
print("=" * 80)

[INFO] INITIALIZING TEST FRAMEWORK
[INFO] All test utility functions initialized successfully.
[INFO] Results list initialized. Current test count: 0
[INFO] Ready to execute test cases.


In [0]:
print('[INFO] Writing raw tables')

record_test("TC07", "Customer raw table exists", lambda: require(has_table(CUSTOMERS_TABLE), CUSTOMERS_TABLE + " missing") or "Table exists")
record_test("TC08", "Product raw table exists", lambda: require(has_table(PRODUCTS_TABLE), PRODUCTS_TABLE + " missing") or "Table exists")
record_test("TC09", "order raw table exists", lambda: require(has_table(order_TABLE), order_TABLE + " missing") or "Table exists")

print('[INFO] Raw tables created: CUSTOMERS_TABLE, PRODUCTS_TABLE, order_TABLE')
print('========== SOURCE LOADING COMPLETE ==========')

[INFO] Writing raw tables

[START] TC07 | Customer raw table exists
[INFO] Table existence check | Table: workspace.default.filter_customer | Exists: True
[PASS] TC07
[DETAILS] Table exists
[END] TC07 | Status: PASS

[START] TC08 | Product raw table exists
[INFO] Table existence check | Table: workspace.default.filter_product | Exists: True
[PASS] TC08
[DETAILS] Table exists
[END] TC08 | Status: PASS

[START] TC09 | order raw table exists
[FAIL] TC09
[ERROR] name 'order_TABLE' is not defined
[END] TC09 | Status: FAIL
[INFO] Raw tables created: CUSTOMERS_TABLE, PRODUCTS_TABLE, order_TABLE
========== SOURCE LOADING COMPLETE ==========


In [0]:
# Cell 3 — Load tables
customers = table_df(CUSTOMERS_TABLE)
products = table_df(PRODUCTS_TABLE)
order = table_df(ORDER_TABLE)
enriched = table_df(ENRICHED_TABLE)

print("Loaded tables")
print("Customers:", customers.count())
print("Products :", products.count())
print("order    :", order.count())
print("Enriched :", enriched.count())

[INFO] Fetching table: workspace.default.filter_customer
[INFO] Table loaded successfully: workspace.default.filter_customer
[INFO] Fetching table: workspace.default.filter_product
[INFO] Table loaded successfully: workspace.default.filter_product
[INFO] Fetching table: workspace.default.filter_order
[INFO] Table loaded successfully: workspace.default.filter_order
[INFO] Fetching table: workspace.default.final_enriched_order
[INFO] Table loaded successfully: workspace.default.final_enriched_order
Loaded tables
Customers: 793
Products : 1851
order    : 9994
Enriched : 10133


**Test cases - 15 to 20. These checks verify that the source paths exist and can be read.Checks path presence and basic readability where practical.**

In [0]:
# Source File Existence Checks (TC15–TC17)

def path_exists(path):
    print(f"[INFO] Checking source path: {path}")

    try:
        print("[INFO] Attempting dbutils.fs.ls()...")
        dbutils.fs.ls(path)

        print(f"[PASS] Path is accessible: {path}")
        return True

    except Exception as ls_exc:
        print(f"[INFO] ls() check failed: {str(ls_exc)[:300]}")
        print("[INFO] Attempting dbutils.fs.head() fallback...")

        try:
            dbutils.fs.head(path, 1)

            print(f"[PASS] Path is readable using head(): {path}")
            return True

        except Exception as head_exc:
            print(f"[FAIL] Path is not accessible: {path}")
            print(f"[ERROR] ls() error: {str(ls_exc)[:300]}")
            print(f"[ERROR] head() error: {str(head_exc)[:300]}")

            return False


print("\n" + "=" * 80)
print("[SECTION START] SOURCE FILE EXISTENCE CHECKS | TC01–TC03")
print("=" * 80)

if CHECK_SOURCE_FILES:

    record_test(
        "TC15",
        "Customer JSON source file exists",
        lambda: (
            require(
                path_exists(CUSTOMERS_FILE),
                f"Customer JSON file not found/readable: {CUSTOMERS_FILE}"
            )
            or "Customer JSON source file exists and is readable"
        )
    )

    record_test(
        "TC16",
        "Product Excel source file exists",
        lambda: (
            require(
                path_exists(PRODUCTS_FILE),
                f"Product Excel file not found/readable: {PRODUCTS_FILE}"
            )
            or "Product Excel source file exists and is readable"
        )
    )

    record_test(
        "TC17",
        "Order CSV source file exists",
        lambda: (
            require(
                path_exists(order_FILE),
                f"Order CSV file not found/readable: {order_FILE}"
            )
            or "Order CSV source file exists and is readable"
        )
    )

else:

    print("[INFO] CHECK_SOURCE_FILES is False")
    print("[INFO] Source existence tests will be skipped.")

    for tid, desc in [
        ("TC15", "Customer JSON source file exists"),
        ("TC16", "Product Excel source file exists"),
        ("TC17", "Order CSV source file exists")
    ]:

        print(f"[SKIP] {tid} | {desc}")

        results.append({
            "Test_ID": tid,
            "Test": desc,
            "Status": "SKIP",
            "Details": "CHECK_SOURCE_FILES=False"
        })

print("[SECTION END] SOURCE FILE EXISTENCE CHECKS | TC15–TC17")
print("=" * 80)


# Format/Readability Checks (TC18–TC20)

def json_readable():

    print("[INFO] Reading Customer JSON source file...")
    print(f"[INFO] Source path: {CUSTOMERS_FILE}")

    test_df = spark.read.json(CUSTOMERS_FILE)

    print("[INFO] JSON file successfully loaded into Spark DataFrame")
    print(f"[INFO] JSON columns: {test_df.columns}")

    require(
        len(test_df.columns) > 0,
        "JSON produced no columns"
    )

    print("[PASS] Customer JSON structure validation completed")

    return f"Readable; columns={test_df.columns}"


def csv_readable():

    print("[INFO] Reading Order CSV source file...")
    print(f"[INFO] Source path: {order_FILE}")
    print("[INFO] CSV options: header=True, inferSchema=False")

    test_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(order_FILE)
    )

    print("[INFO] CSV file successfully loaded into Spark DataFrame")
    print(f"[INFO] CSV columns: {test_df.columns}")

    require(
        len(test_df.columns) > 0,
        "CSV produced no columns"
    )

    print("[PASS] Order CSV structure validation completed")

    return f"Readable; columns={test_df.columns}"


def excel_readable():

    print("[INFO] Reading Product Excel source file...")
    print(f"[INFO] Source path: {PRODUCTS_FILE}")

    import pandas as pd

    print("[INFO] Using pandas read_excel() for Excel validation")

    pdf = pd.read_excel(PRODUCTS_FILE, nrows=5)

    print("[INFO] Excel file successfully loaded")
    print(f"[INFO] Excel columns: {list(pdf.columns)}")

    require(
        len(pdf.columns) > 0,
        "Excel produced no columns"
    )

    print("[PASS] Product Excel structure validation completed")

    return f"Readable; columns={list(pdf.columns)}"


print("\n" + "=" * 80)
print("[SECTION START] FORMAT AND READABILITY CHECKS | TC04–TC06")
print("=" * 80)

if CHECK_SOURCE_FILES:

    record_test(
        "TC18",
        "Customer JSON format/readability",
        lambda: json_readable()
    )

    record_test(
        "TC19",
        "Product Excel format/readability",
        lambda: excel_readable()
    )

    record_test(
        "TC20",
        "Order CSV structure/readability",
        lambda: csv_readable()
    )

else:

    print("[INFO] CHECK_SOURCE_FILES is False")
    print("[INFO] Format/readability tests will be skipped.")

    for tid, desc in [
        ("TC18", "Customer JSON format/readability"),
        ("TC19", "Product Excel format/readability"),
        ("TC20", "Order CSV structure/readability")
    ]:

        print(f"[SKIP] {tid} | {desc}")

        results.append({
            "Test_ID": tid,
            "Test": desc,
            "Status": "SKIP",
            "Details": "CHECK_SOURCE_FILES=False"
        })

print("[SECTION END] FORMAT AND READABILITY CHECKS | TC18–TC20")
print("=" * 80)


[SECTION START] SOURCE FILE EXISTENCE CHECKS | TC01–TC03

[START] TC15 | Customer JSON source file exists
[INFO] Checking source path: /Volumes/workspace/default/pei/customers.json
[INFO] Attempting dbutils.fs.ls()...
[INFO] ls() check failed: (com.databricks.sql.io.CloudFileNotFoundException) No such file or directory /Volumes/workspace/default/pei/customers.json

JVM stacktrace:
com.databricks.sql.io.CloudFileNotFoundException
	at com.databricks.sql.io.NativeIOUtils.translateToSQLStateExceptions(NativeIOUtils.java:466)
	at com.databrick
[INFO] Attempting dbutils.fs.head() fallback...
[FAIL] Path is not accessible: /Volumes/workspace/default/pei/customers.json
[ERROR] ls() error: (com.databricks.sql.io.CloudFileNotFoundException) No such file or directory /Volumes/workspace/default/pei/customers.json

JVM stacktrace:
com.databricks.sql.io.CloudFileNotFoundException
	at com.databricks.sql.io.NativeIOUtils.translateToSQLStateExceptions(NativeIOUtils.java:466)
	at com.databrick
[ERROR] 

In [0]:
# Source File Existence Checks (TC21–TC23)

def path_exists(path):

    print(f"[INFO] Starting file existence check: {path}")

    try:
        print("[INFO] Attempting dbutils.fs.ls()...")
        dbutils.fs.ls(path)

        print(f"[PASS] File/path exists and is accessible: {path}")
        return True

    except Exception as ls_exc:

        print(f"[WARN] dbutils.fs.ls() failed: {str(ls_exc)[:300]}")
        print("[INFO] Attempting dbutils.fs.head() fallback...")

        try:
            dbutils.fs.head(path, 1)

            print(f"[PASS] File is readable using head(): {path}")
            return True

        except Exception as head_exc:

            print(f"[FAIL] File is not accessible: {path}")
            print(f"[ERROR] ls() error: {str(ls_exc)[:300]}")
            print(f"[ERROR] head() error: {str(head_exc)[:300]}")

            return False


print("\n" + "=" * 80)
print("[SECTION START] SOURCE FILE EXISTENCE CHECKS | TC01–TC03")
print("=" * 80)

if CHECK_SOURCE_FILES:

    record_test(
        "TC21",
        "Customer JSON source file exists",
        lambda: (
            require(
                path_exists(CUSTOMERS_FILE),
                f"Not found/readable: {CUSTOMERS_FILE}"
            )
            or "Customer JSON source file exists/readable"
        )
    )

    record_test(
        "TC22",
        "Product Excel source file exists",
        lambda: (
            require(
                path_exists(PRODUCTS_FILE),
                f"Not found/readable: {PRODUCTS_FILE}"
            )
            or "Product Excel source file exists/readable"
        )
    )

    record_test(
        "TC23",
        "Order CSV source file exists",
        lambda: (
            require(
                path_exists(order_FILE),
                f"Not found/readable: {order_FILE}"
            )
            or "Order CSV source file exists/readable"
        )
    )

else:

    print("[INFO] CHECK_SOURCE_FILES=False")
    print("[INFO] Skipping source file existence tests.")

    for tid, desc in [
        ("TC21", "Customer JSON source file exists"),
        ("TC22", "Product Excel source file exists"),
        ("TC23", "Order CSV source file exists")
    ]:

        print(f"[SKIP] {tid} | {desc} | CHECK_SOURCE_FILES=False")

        results.append({
            "Test_ID": tid,
            "Test": desc,
            "Status": "SKIP",
            "Details": "CHECK_SOURCE_FILES=False"
        })

print("[SECTION END] SOURCE FILE EXISTENCE CHECKS | TC01–TC03")
print("=" * 80)


[SECTION START] SOURCE FILE EXISTENCE CHECKS | TC01–TC03

[START] TC21 | Customer JSON source file exists
[INFO] Starting file existence check: /Volumes/workspace/default/pei/customers.json
[INFO] Attempting dbutils.fs.ls()...
[WARN] dbutils.fs.ls() failed: (com.databricks.sql.io.CloudFileNotFoundException) No such file or directory /Volumes/workspace/default/pei/customers.json

JVM stacktrace:
com.databricks.sql.io.CloudFileNotFoundException
	at com.databricks.sql.io.NativeIOUtils.translateToSQLStateExceptions(NativeIOUtils.java:466)
	at com.databrick
[INFO] Attempting dbutils.fs.head() fallback...
[FAIL] File is not accessible: /Volumes/workspace/default/pei/customers.json
[ERROR] ls() error: (com.databricks.sql.io.CloudFileNotFoundException) No such file or directory /Volumes/workspace/default/pei/customers.json

JVM stacktrace:
com.databricks.sql.io.CloudFileNotFoundException
	at com.databricks.sql.io.NativeIOUtils.translateToSQLStateExceptions(NativeIOUtils.java:466)
	at com.data

In [0]:
#Source vs Raw Record Count Reconciliation (TC24–TC26)

def customer_count_reconcile():

    print("[INFO] Starting Customer record count reconciliation")
    print(f"[INFO] Customer source file: {CUSTOMERS_FILE}")

    print("[INFO] Reading Customer source JSON...")
    source = spark.read.json(CUSTOMERS_FILE).count()

    print(f"[INFO] Customer source record count: {source}")

    print("[INFO] Counting records in raw Customer table...")
    raw = customers.count()

    print(f"[INFO] Raw Customer record count: {raw}")

    difference = source - raw

    print(f"[INFO] Customer count difference: {difference}")

    require(
        source == raw,
        f"Customer count mismatch | Source={source} | Raw={raw} | Difference={difference}"
    )

    print("[PASS] Customer source and raw record counts match")

    return f"Matched: Source={source}, Raw={raw}, Difference={difference}"


def product_count_reconcile():

    print("[INFO] Starting Product record count reconciliation")
    print(f"[INFO] Product source file: {PRODUCTS_FILE}")

    import pandas as pd

    print("[INFO] Reading Product source Excel...")
    source = len(pd.read_excel(PRODUCTS_FILE))

    print(f"[INFO] Product source record count: {source}")

    print("[INFO] Counting records in raw Product table...")
    raw = products.count()

    print(f"[INFO] Raw Product record count: {raw}")

    difference = source - raw

    print(f"[INFO] Product count difference: {difference}")

    require(
        source == raw,
        f"Product count mismatch | Source={source} | Raw={raw} | Difference={difference}"
    )

    print("[PASS] Product source and raw record counts match")

    return f"Matched: Source={source}, Raw={raw}, Difference={difference}"


def order_count_reconcile():

    print("[INFO] Starting Order record count reconciliation")
    print(f"[INFO] Order source file: {order_FILE}")

    print("[INFO] Reading Order source CSV...")
    source = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(order_FILE)
        .count()
    )

    print(f"[INFO] Order source record count: {source}")

    print("[INFO] Counting records in raw Order table...")
    raw = order.count()

    print(f"[INFO] Raw Order record count: {raw}")

    difference = source - raw

    print(f"[INFO] Order count difference: {difference}")

    require(
        source == raw,
        f"Order count mismatch | Source={source} | Raw={raw} | Difference={difference}"
    )

    print("[PASS] Order source and raw record counts match")

    return f"Matched: Source={source}, Raw={raw}, Difference={difference}"


print("\n" + "=" * 80)
print("[SECTION START] SOURCE VS RAW RECORD COUNT RECONCILIATION | TC10–TC12")
print("=" * 80)

if CHECK_SOURCE_FILES:

    record_test(
        "TC24",
        "Customer source vs raw record count",
        customer_count_reconcile
    )

    record_test(
        "TC25",
        "Product source vs raw record count",
        product_count_reconcile
    )

    record_test(
        "TC26",
        "Order source vs raw record count",
        order_count_reconcile
    )

else:

    print("[INFO] CHECK_SOURCE_FILES=False")
    print("[INFO] Record count reconciliation tests will be skipped.")

    for tid, desc in [
        ("TC24", "Customer source vs raw record count"),
        ("TC25", "Product source vs raw record count"),
        ("TC26", "Order source vs raw record count")
    ]:

        print(f"[SKIP] {tid} | {desc}")

        results.append({
            "Test_ID": tid,
            "Test": desc,
            "Status": "SKIP",
            "Details": "CHECK_SOURCE_FILES=False"
        })

print("[SECTION END] SOURCE VS RAW RECORD COUNT RECONCILIATION | TC10–TC12")
print("=" * 80)


[SECTION START] SOURCE VS RAW RECORD COUNT RECONCILIATION | TC10–TC12

[START] TC24 | Customer source vs raw record count
[INFO] Starting Customer record count reconciliation
[INFO] Customer source file: /Volumes/workspace/default/pei/customers.json
[INFO] Reading Customer source JSON...
[FAIL] TC24
[ERROR] [PATH_NOT_FOUND] Path does not exist: dbfs:/Volumes/workspace/default/pei/customers.json. SQLSTATE: 42K03

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.dataPathNotExistError(QueryCompilationErrors.scala:3651)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:1291)
	at scala.collection.immutable.List.flatMap(List.scala:283)
	at scala.collection.immutable.List.flatMap(
[END] TC24 | Status: FAIL

[START] TC25 | Product source vs raw record count
[INFO] Starting Product record count reconciliation
[INFO] Product source file: /Volumes/workspace/default/pei/p

In [0]:
# Customer Quality Checks (TC27–TC36)
print("\n" + "=" * 100)
print("[SECTION START] CUSTOMER DATA QUALITY CHECKS | TC27–TC36")
print("=" * 100)

# -----------------------------------------------------------------------------
# Customer schema validation
# -----------------------------------------------------------------------------

print("\n[INFO] Validating Customer table schema...")
print(f"[INFO] Customer table columns: {customers.columns}")

record_test(
    "TC_SCHEMA_C",
    "Customer table required columns validation",
    lambda: require_columns(
        customers,
        [CUSTOMER_ID, CUSTOMER_NAME, COUNTRY],
        "Customer table"
    ) or "Required columns validated"
)


# -----------------------------------------------------------------------------
# TC27 — CustomerID is not NULL
# -----------------------------------------------------------------------------

def customer_id_null_check():
    print(f"[INFO] Checking NULL values in {CUSTOMER_ID}")

    count = null_count(customers, CUSTOMER_ID)

    require(count == 0, f"CustomerID NULL count={count}")

    return f"NULL count={count}"


record_test(
    "TC27",
    "CustomerID is not NULL",
    customer_id_null_check
)


# -----------------------------------------------------------------------------
# TC28 — CustomerID uniqueness
# -----------------------------------------------------------------------------

def customer_id_uniqueness_check():
    print(f"[INFO] Checking duplicate CustomerID groups for {CUSTOMER_ID}")

    count = duplicate_key_count(customers, CUSTOMER_ID)

    require(count == 0, f"Duplicate CustomerID groups={count}")

    return f"Duplicate ID groups={count}"


record_test(
    "TC28",
    "CustomerID uniqueness",
    customer_id_uniqueness_check
)


# -----------------------------------------------------------------------------
# TC29 — Customer_Name is not NULL
# -----------------------------------------------------------------------------

def customer_name_null_check():
    print(f"[INFO] Checking NULL values in {CUSTOMER_NAME}")

    count = null_count(customers, CUSTOMER_NAME)

    require(count == 0, f"Customer_Name NULL count={count}")

    return f"NULL count={count}"


record_test(
    "TC29",
    "Customer_Name is not NULL",
    customer_name_null_check
)


# -----------------------------------------------------------------------------
# TC30 — Customer_Name is not blank
# -----------------------------------------------------------------------------

def customer_name_blank_check():
    print(f"[INFO] Checking blank values in {CUSTOMER_NAME}")

    count = blank_count(customers, CUSTOMER_NAME)

    require(count == 0, f"Customer_Name blank count={count}")

    return f"Blank count={count}"

record_test(
    "TC30",
    "Customer_Name is not blank",
    customer_name_blank_check
)


# -----------------------------------------------------------------------------
# TC31 — Customer_Name suspicious patterns
# -----------------------------------------------------------------------------

def suspicious_names():

    print("[INFO] Checking suspicious/formula-like patterns in Customer_Name")
    print("[INFO] Pattern check includes '=', '+', '@' and '--' prefixes")

    bad = customers.filter(
        F.col(CUSTOMER_NAME).cast("string").rlike(r"^\s*[=+@]|^\s*--")
    ).count()

    print(f"[INFO] Suspicious name rows detected: {bad}")

    require(
        bad == 0,
        f"Suspicious name rows={bad}; review names resembling formula/injection patterns"
    )

    return f"Suspicious rows={bad}"


record_test(
    "TC31",
    "Customer_Name suspicious patterns",
    suspicious_names
)

# -----------------------------------------------------------------------------
# TC32 — Country is not NULL
# -----------------------------------------------------------------------------

def country_null_check():

    print(f"[INFO] Checking NULL values in {COUNTRY}")

    count = null_count(customers, COUNTRY)

    require(count == 0, f"Country NULL count={count}")

    return f"NULL count={count}"


record_test(
    "TC32",
    "Country is not NULL",
    country_null_check
)

# -----------------------------------------------------------------------------
# TC33 — Country is not blank
# -----------------------------------------------------------------------------

def country_blank_check():

    print(f"[INFO] Checking blank values in {COUNTRY}")

    count = blank_count(customers, COUNTRY)

    require(count == 0, f"Country blank count={count}")

    return f"Blank count={count}"


record_test(
    "TC33",
    "Country is not blank",
    country_blank_check
)


# -----------------------------------------------------------------------------
# TC34 — Country format sanity check
# -----------------------------------------------------------------------------

def country_format_check():

    print("[INFO] Performing Country format sanity validation")
    print("[INFO] Allowed pattern: alphabets, spaces, periods, apostrophes and hyphens")

    invalid = customers.filter(
        ~F.col(COUNTRY).cast("string").rlike(r"^[A-Za-z][A-Za-z .'-]*$")
    ).count()

    print(f"[INFO] Country format anomalies detected: {invalid}")

    require(
        invalid == 0,
        f"Country format anomalies={invalid}"
    )

    return f"Anomalies={invalid}"


record_test(
    "TC34",
    "Country format sanity check",
    country_format_check
)


# -----------------------------------------------------------------------------
# TC35 — Exact duplicate customer records
# -----------------------------------------------------------------------------

def duplicate_customer_rows():

    print("[INFO] Checking exact duplicate Customer records")
    print("[INFO] Grouping by all Customer table columns")

    dup = (
        customers.groupBy(*customers.columns)
        .count()
        .filter("count > 1")
        .count()
    )

    print(f"[INFO] Exact duplicate row groups detected: {dup}")

    require(
        dup == 0,
        f"Exact duplicate row groups={dup}"
    )

    return f"Exact duplicate groups={dup}"


record_test(
    "TC35",
    "Exact duplicate customer records",
    duplicate_customer_rows
)


# -----------------------------------------------------------------------------
# TC36 — CustomerID data type sanity
# -----------------------------------------------------------------------------

def customer_id_type():

    print(f"[INFO] Checking data type for {CUSTOMER_ID}")

    dtype = dict(customers.dtypes)[CUSTOMER_ID]

    print(f"[INFO] CustomerID data type detected: {dtype}")

    require(
        dtype in ("string", "int", "bigint", "long"),
        f"Unexpected CustomerID type={dtype}"
    )

    return f"Type={dtype}"

record_test(
    "TC36",
    "CustomerID data type sanity",
    customer_id_type
)

print("\n" + "=" * 100)
print("[SECTION END] CUSTOMER DATA QUALITY CHECKS | TC13–TC22")
print("[INFO] Customer quality test execution completed.")
print("=" * 100)


[SECTION START] CUSTOMER DATA QUALITY CHECKS | TC27–TC36

[INFO] Validating Customer table schema...
[INFO] Customer table columns: ['Customer_ID', 'Customer_Name', 'email', 'phone', 'address', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region']

[START] TC_SCHEMA_C | Customer table required columns validation
[INFO] Validating columns for: Customer table
[INFO] Expected columns: ['Customer_ID', 'Customer_Name', 'Country']
[PASS] All required columns exist for Customer table
[PASS] TC_SCHEMA_C
[DETAILS] Required columns validated
[END] TC_SCHEMA_C | Status: PASS

[START] TC27 | CustomerID is not NULL
[INFO] Checking NULL values in Customer_ID
[INFO] Checking NULL values in column: Customer_ID
[FAIL] TC27
[ERROR] name 'F' is not defined
[END] TC27 | Status: FAIL

[START] TC28 | CustomerID uniqueness
[INFO] Checking duplicate CustomerID groups for Customer_ID
[INFO] Checking duplicate keys: Customer_ID
[FAIL] TC28
[ERROR] name 'F' is not defined
[END] TC28 | Status: FAIL

[S

In [0]:
# =============================================================================
# Section D — Product Quality Checks (TC37–TC45)
# =============================================================================

print("\n" + "=" * 100)
print("[SECTION START] PRODUCT DATA QUALITY CHECKS | TC37–TC45")
print("=" * 100)


# -----------------------------------------------------------------------------
# Product schema validation
# -----------------------------------------------------------------------------

print("\n[INFO] Validating Product table schema...")
print(f"[INFO] Product table columns: {products.columns}")

record_test(
    "TC_SCHEMA_D",
    "Product table required columns validation",
    lambda: require_columns(
        products,
        [PRODUCT_ID, PRODUCT_NAME, CATEGORY, SUBCATEGORY],
        "Product table"
    ) or "Required columns validated"
)


# -----------------------------------------------------------------------------
# TC37 — ProductID is not NULL
# -----------------------------------------------------------------------------

def product_id_null_check():

    print(f"[INFO] Checking NULL values in {PRODUCT_ID}")

    count = null_count(products, PRODUCT_ID)

    require(count == 0, f"ProductID NULL count={count}")

    return f"NULL count={count}"


record_test(
    "TC37",
    "ProductID is not NULL",
    product_id_null_check
)


# -----------------------------------------------------------------------------
# TC38 — ProductID uniqueness
# -----------------------------------------------------------------------------

def product_id_uniqueness_check():

    print(f"[INFO] Checking duplicate ProductID groups for {PRODUCT_ID}")

    count = duplicate_key_count(products, PRODUCT_ID)

    require(count == 0, f"Duplicate ProductID groups={count}")

    return f"Duplicate ID groups={count}"


record_test(
    "TC38",
    "ProductID uniqueness",
    product_id_uniqueness_check
)


# -----------------------------------------------------------------------------
# TC39 — Product_Name is not NULL
# -----------------------------------------------------------------------------

def product_name_null_check():

    print(f"[INFO] Checking NULL values in {PRODUCT_NAME}")

    count = null_count(products, PRODUCT_NAME)

    require(count == 0, f"Product_Name NULL count={count}")

    return f"NULL count={count}"


record_test(
    "TC39",
    "Product_Name is not NULL",
    product_name_null_check
)


# -----------------------------------------------------------------------------
# TC40 — Product_Name is not blank
# -----------------------------------------------------------------------------

def product_name_blank_check():

    print(f"[INFO] Checking blank values in {PRODUCT_NAME}")

    count = blank_count(products, PRODUCT_NAME)

    require(count == 0, f"Product_Name blank count={count}")

    return f"Blank count={count}"


record_test(
    "TC40",
    "Product_Name is not blank",
    product_name_blank_check
)


# -----------------------------------------------------------------------------
# TC41 — Category is not NULL
# -----------------------------------------------------------------------------

def category_null_check():

    print(f"[INFO] Checking NULL values in {CATEGORY}")

    count = null_count(products, CATEGORY)

    require(count == 0, f"Category NULL count={count}")

    return f"NULL count={count}"


record_test(
    "TC41",
    "Category is not NULL",
    category_null_check
)


# -----------------------------------------------------------------------------
# TC42 — SubCategory is not NULL
# -----------------------------------------------------------------------------

def subcategory_null_check():

    print(f"[INFO] Checking NULL values in {SUBCATEGORY}")

    count = null_count(products, SUBCATEGORY)

    require(count == 0, f"SubCategory NULL count={count}")

    return f"NULL count={count}"


record_test(
    "TC42",
    "Sub-Category is not NULL",
    subcategory_null_check
)


# -----------------------------------------------------------------------------
# TC43 — Category value sanity
# -----------------------------------------------------------------------------

def category_values_check():

    print("[INFO] Performing Category value sanity check")
    print("[INFO] Checking for blank Category values")

    invalid = products.filter(
        F.trim(F.col(CATEGORY).cast("string")) == ""
    ).count()

    print(f"[INFO] Blank category rows detected: {invalid}")

    require(
        invalid == 0,
        f"Blank category rows={invalid}"
    )

    return f"Blank category rows={invalid}"


record_test(
    "TC43",
    "Category value sanity",
    category_values_check
)


# -----------------------------------------------------------------------------
# TC44 — SubCategory-to-Category mapping consistency
# -----------------------------------------------------------------------------

def subcategory_mapping_check():

    print("[INFO] Checking SubCategory-to-Category mapping consistency")
    print("[INFO] Checking whether one SubCategory maps to multiple Categories")

    conflicts = (
        products.groupBy(SUBCATEGORY)
        .agg(F.countDistinct(CATEGORY).alias("category_count"))
        .filter(F.col("category_count") > 1)
        .count()
    )

    print(f"[INFO] SubCategory mapping conflicts detected: {conflicts}")

    require(
        conflicts == 0,
        f"SubCategory-to-Category conflicts={conflicts}"
    )

    return f"Conflicting subcategories={conflicts}"


record_test(
    "TC44",
    "SubCategory-to-Category mapping consistency",
    subcategory_mapping_check
)


# -----------------------------------------------------------------------------
# TC45 — ProductID data type sanity
# -----------------------------------------------------------------------------

def product_id_type():

    print(f"[INFO] Checking data type for {PRODUCT_ID}")

    dtype = dict(products.dtypes)[PRODUCT_ID]

    print(f"[INFO] ProductID data type detected: {dtype}")

    require(
        dtype in ("string", "int", "bigint", "long"),
        f"Unexpected ProductID type={dtype}"
    )

    return f"Type={dtype}"


record_test(
    "TC45",
    "ProductID data type sanity",
    product_id_type
)


print("\n" + "=" * 100)
print("[SECTION END] PRODUCT DATA QUALITY CHECKS | TC23–TC31")
print("[INFO] Product quality test execution completed.")
print("=" * 100)


[SECTION START] PRODUCT DATA QUALITY CHECKS | TC37–TC45

[INFO] Validating Product table schema...
[INFO] Product table columns: ['Product_ID', 'Category', 'Sub-Category', 'Product_Name', 'State', 'Price_per_product']

[START] TC_SCHEMA_D | Product table required columns validation
[INFO] Validating columns for: Product table
[INFO] Expected columns: ['Product_ID', 'Product_Name', 'Category', 'Sub-Category']
[PASS] All required columns exist for Product table
[PASS] TC_SCHEMA_D
[DETAILS] Required columns validated
[END] TC_SCHEMA_D | Status: PASS

[START] TC37 | ProductID is not NULL
[INFO] Checking NULL values in Product_ID
[INFO] Checking NULL values in column: Product_ID
[FAIL] TC37
[ERROR] name 'F' is not defined
[END] TC37 | Status: FAIL

[START] TC38 | ProductID uniqueness
[INFO] Checking duplicate ProductID groups for Product_ID
[INFO] Checking duplicate keys: Product_ID
[FAIL] TC38
[ERROR] name 'F' is not defined
[END] TC38 | Status: FAIL

[START] TC39 | Product_Name is not NU

In [0]:

print("\n" + "="*80)
print("SECTION — ENRICHED ORDER VALIDATION (TC46–TC52)")
print("="*80)

print("\n[INFO] Validating required columns in Enriched table...")
print(f"[INFO] Required columns: {ORDER_DATE}, {PROFIT_COL}, {CUSTOMER_NAME}, {COUNTRY}, {CATEGORY}, {SUBCATEGORY}")

record_test(
    "TC_SCHEMA_F",
    "Enriched table required columns",
    lambda: require_columns(
        enriched,
        [ORDER_DATE, PROFIT_COL, CUSTOMER_NAME, COUNTRY, CATEGORY, SUBCATEGORY],
        "Enriched table"
    ) or "Required columns validated"
)

# TC46 — Enriched table existence
print("\n[INFO] TC46 — Checking enriched order table existence...")

record_test(
    "TC46",
    "Enriched order table exists",
    lambda: require(has_table(ENRICHED_TABLE), ENRICHED_TABLE+" missing") or "Table exists"
)

# TC47 — Enriched Customer_Name quality
print("\n[INFO] TC47 — Validating Customer_Name in enriched table...")

def enriched_customer_name_check():
    print("[INFO] Checking NULL, blank and suspicious Customer_Name values...")

    bad = enriched.filter(
        F.col(CUSTOMER_NAME).isNull() |
        (F.trim(F.col(CUSTOMER_NAME))=="") |
        F.col(CUSTOMER_NAME).rlike(r"^\s*[=+@]|^\s*--")
    ).count()

    print(f"[INFO] Invalid Customer_Name rows={bad}")

    require(bad==0, f"Invalid/null/blank/suspicious Customer_Name rows={bad}")

    return f"Invalid rows={bad}"

record_test("TC47", "Enriched Customer_Name quality", enriched_customer_name_check)

# TC48 — Enriched Country quality
print("\n[INFO] TC48 — Validating Country in enriched table...")

record_test(
    "TC48",
    "Enriched Country quality",
    lambda: require(
        null_count(enriched, COUNTRY)==0 and blank_count(enriched, COUNTRY)==0,
        f"Null={null_count(enriched,COUNTRY)}, blank={blank_count(enriched,COUNTRY)}"
    ) or "Country clean"
)

# TC49 — Enriched Category quality
print("\n[INFO] TC49 — Validating Category in enriched table...")

record_test(
    "TC49",
    "Enriched Category quality",
    lambda: require(
        null_count(enriched, CATEGORY)==0 and blank_count(enriched, CATEGORY)==0,
        f"Null={null_count(enriched,CATEGORY)}, blank={blank_count(enriched,CATEGORY)}"
    ) or "Category clean"
)

# TC50 — Enriched SubCategory quality
print("\n[INFO] TC47 — Validating SubCategory in enriched table...")

record_test(
    "TC50",
    "Enriched SubCategory quality",
    lambda: require(
        null_count(enriched, SUBCATEGORY)==0 and blank_count(enriched, SUBCATEGORY)==0,
        f"Null={null_count(enriched,SUBCATEGORY)}, blank={blank_count(enriched,SUBCATEGORY)}"
    ) or "SubCategory clean"
)

# TC51 — Profit rounded to 2 decimals
print("\n[INFO] TC51 — Checking Profit rounding to 2 decimal places...")

def profit_rounded_check():
    print("[INFO] Comparing Profit values with rounded 2-decimal values...")

    invalid = enriched.filter(
        F.col(PROFIT_COL).cast("double") !=
        F.round(F.col(PROFIT_COL).cast("double"),2)
    ).count()

    print(f"[INFO] Profit values not rounded to 2 decimals={invalid}")

    require(invalid==0, f"Profit values not rounded to 2 decimal places={invalid}")

    return f"Unrounded rows={invalid}"

record_test("TC51", "Enriched Profit rounded to 2 decimals", profit_rounded_check)

# TC52 — Enriched row count reconciliation
print("\n[INFO] TC52 — Reconciling enriched row count with expected inner joins...")

def enriched_count_reconcile():
    print("[INFO] Validating join key columns...")

    require_columns(order, [CUSTOMER_ID, PRODUCT_ID], "order table for reconciliation")
    require_columns(customers, [CUSTOMER_ID], "Customer table for reconciliation")
    require_columns(products, [PRODUCT_ID], "Product table for reconciliation")

    print("[INFO] Performing expected inner joins...")

    expected = order.join(customers, CUSTOMER_ID, "inner").join(products, PRODUCT_ID, "inner").count()

    print(f"[INFO] Expected inner-join row count={expected}")

    print("[INFO] Counting actual enriched table rows...")

    actual = enriched.count()

    print(f"[INFO] Actual enriched row count={actual}")
    print(f"[INFO] Row count difference={actual-expected}")

    require(actual==expected, f"Expected inner-join count={expected}; enriched count={actual}")

    return f"Expected={expected}; actual={actual}"

record_test("TC52", "Enriched row count reconciliation", enriched_count_reconcile)

print("\n" + "="*80)
print("SECTION — ENRICHED ORDER VALIDATION COMPLETED")
print("="*80)


SECTION F — ENRICHED ORDER VALIDATION (TC43–TC49)

[INFO] Validating required columns in Enriched table...
[INFO] Required columns: Order_Date, Profit, Customer_Name, Country, Category, Sub-Category

[START] TC_SCHEMA_F | Enriched table required columns
[INFO] Validating columns for: Enriched table
[INFO] Expected columns: ['Order_Date', 'Profit', 'Customer_Name', 'Country', 'Category', 'Sub-Category']
[PASS] All required columns exist for Enriched table
[PASS] TC_SCHEMA_F
[DETAILS] Required columns validated
[END] TC_SCHEMA_F | Status: PASS

[INFO] TC46 — Checking enriched order table existence...

[START] TC46 | Enriched order table exists
[INFO] Table existence check | Table: workspace.default.final_enriched_order | Exists: True
[PASS] TC46
[DETAILS] Table exists
[END] TC46 | Status: PASS

[INFO] TC47 — Validating Customer_Name in enriched table...

[START] TC47 | Enriched Customer_Name quality
[INFO] Checking NULL, blank and suspicious Customer_Name values...
[FAIL] TC47
[ERROR] n

In [0]:
# ## Section — Business logic and aggregations (TC53–TC58)

import pyspark.sql.functions as F

# These tests/reporting checks treat negative profit as potentially valid.
# The aggregate negative-profit check is a business warning.

print("\n" + "="*80)
print("SECTION — BUSINESS LOGIC AND AGGREGATIONS (TC53–TC58)")
print("="*80)

# TC53 — Negative individual Profit report
print("\n[INFO] TC53 — Reporting individual negative Profit values...")
print("[INFO] Negative Profit is treated as valid business data, not a failure.")

def negative_profit_report():

    print("[INFO] Validating required columns...")
    require_columns(enriched, [PROFIT_COL], "Enriched for negative Profit report")

    print("[INFO] Counting individual negative Profit rows...")

    count = enriched.filter(
        F.col(PROFIT_COL).cast("double") < 0
    ).count()

    print(f"[INFO] Individual negative Profit rows={count}")
    print("[INFO] Negative Profit rows are treated as valid business data, not a failure.")

    if count > 0:
        print(f"[WARNING] {count} rows have negative Profit; investigate business context.")

    return f"Negative Profit rows={count}; treated as valid business data"

record_test("TC53", "Negative individual Profit report", negative_profit_report)


# TC54 — Total Profit reconciliation
print("\n[INFO] TC54 — Reconciling total Profit between raw and enriched tables...")

def total_profit_reconcile():

    print("[INFO] Calculating total Profit from raw order table...")

    raw_total = order.agg(
        F.round(F.sum(F.col(PROFIT_COL).cast("double")),2).alias("total")
    ).first()["total"]

    print(f"[INFO] Raw total Profit={raw_total}")

    print("[INFO] Calculating total Profit from enriched table...")

    enriched_total = enriched.agg(
        F.round(F.sum(F.col(PROFIT_COL).cast("double")),2).alias("total")
    ).first()["total"]

    print(f"[INFO] Enriched total Profit={enriched_total}")

    print(f"[INFO] Difference={None if raw_total is None or enriched_total is None else enriched_total-raw_total}")

    require(
        raw_total == enriched_total,
        f"Raw total={raw_total}; enriched total={enriched_total}"
    )

    return f"Matched total={raw_total}"

record_test("TC54", "Total Profit reconciliation", total_profit_reconcile)


# TC55 — Negative aggregate Profit report
print("\n[INFO] TC52 — Checking aggregate total Profit...")
print("[INFO] This check generates a business warning only; it does not fail.")

def negative_total_profit_report():

    print("[INFO] Calculating aggregate Profit from enriched table...")

    total = enriched.agg(
        F.sum(F.col(PROFIT_COL).cast("double")).alias("total")
    ).first()["total"]

    print(f"[INFO] Aggregate total Profit={total}")

    if total is None:
        print("[WARNING] Total Profit is NULL (no records)")
        return "Total profit is NULL (no records)"

    if total < 0:
        print(f"[WARNING] Aggregate total profit is negative ({total})")
        print("[WARNING] Investigate business context.")
        return f"WARNING: aggregate total profit is negative ({total}); investigate business context"

    print("[INFO] Aggregate total Profit is not negative.")

    return f"Aggregate total profit={total}; not negative"

record_test("TC55", "Negative aggregate Profit report (warning only)", negative_total_profit_report)


# TC56 — Profit by Year aggregation
print("\n[INFO] TC56 — Validating Profit by Year aggregation...")

def profit_by_year_reconcile():

    print("[INFO] Validating required columns...")
    require_columns(enriched, [ORDER_DATE, PROFIT_COL], "Enriched for Year aggregation")

    print("[INFO] Parsing OrderDate...")
    parsed = enriched.withColumn(
        "_date",
        F.to_date(F.col(ORDER_DATE).cast("string"), "d/M/yyyy")
    )

    print("[INFO] Computing Profit by Year...")
    by_year = parsed.groupBy(
        F.year("_date").alias("year")
    ).agg(
        F.round(F.sum(F.col(PROFIT_COL).cast("double")),2).alias("profit")
    )

    print("[INFO] Computing independent comparison aggregation...")
    check = parsed.groupBy(
        F.year("_date").alias("year")
    ).agg(
        F.round(F.sum(F.col(PROFIT_COL).cast("double")),2).alias("profit")
    )

    print("[INFO] Comparing both aggregation outputs...")

    mismatch_1 = by_year.exceptAll(check).count()
    mismatch_2 = check.exceptAll(by_year).count()

    print(f"[INFO] Mismatch rows (by_year vs check)={mismatch_1}")
    print(f"[INFO] Mismatch rows (check vs by_year)={mismatch_2}")

    require(
        mismatch_1==0 and mismatch_2==0,
        "Year aggregation mismatch"
    )

    year_count = by_year.count()

    print(f"[INFO] Year groups validated={year_count}")

    print("[INFO] Profit by Year aggregation validation completed.")

    return f"Years validated={year_count}"

record_test("TC56", "Profit by Year aggregation", profit_by_year_reconcile)


# TC57 — Profit by Year + Category aggregation
print("\n[INFO] TC57 — Recomputing Profit by Year + Category...")

def profit_year_category_report():

    print("[INFO] Validating required columns...")
    require_columns(
        enriched,
        [ORDER_DATE, CATEGORY, PROFIT_COL],
        "Enriched for Year/Category aggregation"
    )

    print("[INFO] Parsing OrderDate...")
    parsed = enriched.withColumn(
        "_date",
        F.to_date(F.col(ORDER_DATE).cast("string"), "d/M/yyyy")
    )

    print("[INFO] Grouping Profit by Year and Category...")
    grouped = parsed.groupBy(
        F.year("_date").alias("year"),
        CATEGORY
    ).agg(
        F.round(F.sum(F.col(PROFIT_COL).cast("double")),2).alias("profit")
    )

    print("[INFO] Counting Year + Category groups...")
    group_count = grouped.count()

    print(f"[INFO] Year + Category groups recomputed={group_count}")

    return f"Year+Category groups validated/recomputed={group_count}"

record_test("TC57", "Profit by Year + Category aggregation", profit_year_category_report)


# TC58 — Profit by Customer and Customer + Year
print("\n[INFO] TC58 — Recomputing Profit by Customer and Customer + Year...")

def profit_customer_and_year_report():

    print("[INFO] Validating required columns...")
    require_columns(
        enriched,
        [CUSTOMER_NAME, ORDER_DATE, PROFIT_COL],
        "Enriched for Customer aggregation"
    )

    print("[INFO] Parsing OrderDate...")
    parsed = enriched.withColumn(
        "_date",
        F.to_date(F.col(ORDER_DATE).cast("string"), "d/M/yyyy")
    )

    print("[INFO] Grouping Profit by Customer...")
    by_customer = parsed.groupBy(
        CUSTOMER_NAME
    ).agg(
        F.round(F.sum(F.col(PROFIT_COL).cast("double")),2).alias("profit")
    )

    print("[INFO] Grouping Profit by Customer + Year...")
    by_customer_year = parsed.groupBy(
        CUSTOMER_NAME,
        F.year("_date").alias("year")
    ).agg(
        F.round(F.sum(F.col(PROFIT_COL).cast("double")),2).alias("profit")
    )

    customer_groups = by_customer.count()
    customer_year_groups = by_customer_year.count()

    print(f"[INFO] Customer groups={customer_groups}")
    print(f"[INFO] Customer + Year groups={customer_year_groups}")

    print("[INFO] Customer aggregations recomputed successfully.")

    return f"Customer groups={customer_groups}; Customer+Year groups={customer_year_groups}"

record_test(
    "TC58",
    "Profit by Customer and Customer + Year aggregation",
    profit_customer_and_year_report
)

print("\n" + "="*80)
print("SECTION G — BUSINESS LOGIC AND AGGREGATIONS COMPLETED")
print("="*80)


SECTION — BUSINESS LOGIC AND AGGREGATIONS (TC53–TC58)

[INFO] TC53 — Reporting individual negative Profit values...
[INFO] Negative Profit is treated as valid business data, not a failure.

[START] TC53 | Negative individual Profit report
[INFO] Validating required columns...
[INFO] Validating columns for: Enriched for negative Profit report
[INFO] Expected columns: ['Profit']
[PASS] All required columns exist for Enriched for negative Profit report
[INFO] Counting individual negative Profit rows...
[INFO] Individual negative Profit rows=1911
[INFO] Negative Profit rows are treated as valid business data, not a failure.
[WARNING] 1911 rows have negative Profit; investigate business context.
[PASS] TC53
[DETAILS] Negative Profit rows=1911; treated as valid business data
[END] TC53 | Status: PASS

[INFO] TC54 — Reconciling total Profit between raw and enriched tables...

[START] TC54 | Total Profit reconciliation
[INFO] Calculating total Profit from raw order table...
[INFO] Raw total P

In [0]:
import pytest
from pyspark.sql import functions as F

# ==========================================
# FIXTURE
# ==========================================

@pytest.fixture
def customer_fixture():
    print("\n[FIXTURE] Loading existing customers DataFrame")
    print("[FIXTURE] Customer table loaded successfully")
    print(f"[FIXTURE] Total customer rows={customers.count()}")
    return customers


# TC01 — CustomerID is not NULL
def test_customer_id_not_null(customer_fixture):
    print("\n[START] TC01 — CustomerID is not NULL")

    null_count = customer_fixture.filter(
        F.col(CUSTOMER_ID).isNull()
    ).count()

    print(f"[INFO] CustomerID NULL count={null_count}")

    assert null_count == 0, f"CustomerID NULL count={null_count}"

    print("[PASS] TC01 — CustomerID is not NULL")


# TC02 — CustomerID uniqueness
def test_customer_id_unique(customer_fixture):
    print("\n[START] TC02 — CustomerID uniqueness")

    duplicate_count = (
        customer_fixture.groupBy(CUSTOMER_ID)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(f"[INFO] Duplicate CustomerID groups={duplicate_count}")

    assert duplicate_count == 0, f"Duplicate CustomerID groups={duplicate_count}"

    print("[PASS] TC02 — CustomerID uniqueness")


# TC03 — Customer_Name is not NULL
def test_customer_name_not_null(customer_fixture):
    print("\n[START] TC03 — Customer_Name is not NULL")

    null_count = customer_fixture.filter(
        F.col(CUSTOMER_NAME).isNull()
    ).count()

    print(f"[INFO] Customer_Name NULL count={null_count}")

    assert null_count == 0, f"Customer_Name NULL count={null_count}"

    print("[PASS] TC03 — Customer_Name is not NULL")


# TC04 — Customer_Name is not blank
def test_customer_name_not_blank(customer_fixture):
    print("\n[START] TC04 — Customer_Name is not blank")

    blank_count = customer_fixture.filter(
        F.trim(F.col(CUSTOMER_NAME)) == ""
    ).count()

    print(f"[INFO] Blank Customer_Name count={blank_count}")

    assert blank_count == 0, f"Blank Customer_Name count={blank_count}"

    print("[PASS] TC04 — Customer_Name is not blank")


# TC05 — Suspicious Customer_Name
def test_customer_name_suspicious_values(customer_fixture):
    print("\n[START] TC05 — Suspicious Customer_Name")

    suspicious_count = customer_fixture.filter(
        F.col(CUSTOMER_NAME).rlike(r"^\s*[=+@]|^\s*--")
    ).count()

    print(f"[INFO] Suspicious Customer_Name count={suspicious_count}")

    assert suspicious_count == 0, f"Suspicious Customer_Name count={suspicious_count}"

    print("[PASS] TC05 — Suspicious Customer_Name")


# TC06 — Country is not NULL
def test_country_not_null(customer_fixture):
    print("\n[START] TC06 — Country is not NULL")

    null_count = customer_fixture.filter(
        F.col(COUNTRY).isNull()
    ).count()

    print(f"[INFO] Country NULL count={null_count}")

    assert null_count == 0, f"Country NULL count={null_count}"

    print("[PASS] TC06 — Country is not NULL")


# TC07 — Country is not blank
def test_country_not_blank(customer_fixture):
    print("\n[START] TC07 — Country is not blank")

    blank_count = customer_fixture.filter(
        F.trim(F.col(COUNTRY)) == ""
    ).count()

    print(f"[INFO] Country blank count={blank_count}")

    assert blank_count == 0, f"Country blank count={blank_count}"

    print("[PASS] TC07 — Country is not blank")


# TC08 — Country format sanity
def test_country_format(customer_fixture):
    print("\n[START] TC08 — Country format sanity")

    invalid_count = customer_fixture.filter(
        ~F.col(COUNTRY).rlike(r"^[A-Za-z][A-Za-z .'-]*$")
    ).count()

    print(f"[INFO] Invalid Country format count={invalid_count}")

    assert invalid_count == 0, f"Invalid Country rows={invalid_count}"

    print("[PASS] TC08 — Country format sanity")


# TC09 — Exact duplicate customer records
def test_exact_duplicate_customer_rows(customer_fixture):
    print("\n[START] TC09 — Exact duplicate customer records")

    duplicate_count = (
        customer_fixture.groupBy(*customer_fixture.columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(f"[INFO] Exact duplicate row groups={duplicate_count}")

    assert duplicate_count == 0, f"Duplicate row groups={duplicate_count}"

    print("[PASS] TC09 — Exact duplicate customer records")


# TC10 — CustomerID data type
def test_customer_id_datatype(customer_fixture):
    print("\n[START] TC10 — CustomerID data type")

    datatype = dict(customer_fixture.dtypes)[CUSTOMER_ID]

    print(f"[INFO] CustomerID datatype={datatype}")

    assert datatype in ("string", "int", "bigint"), f"Unexpected datatype={datatype}"

    print("[PASS] TC10 — CustomerID data type")


# ==========================================
# FINAL SECTION LOG
# ==========================================

print("\n" + "="*70)
print("[INFO] CUSTOMER FIXTURE TESTS TC01–TC10 DEFINED")
print("[INFO] Ready for PyTest execution")
print("="*70)


[INFO] CUSTOMER FIXTURE TESTS TC01–TC10 DEFINED
[INFO] Ready for PyTest execution


**We required only encriched table as it is having all transformed data**

In [0]:
%sql
select * from final_enriched_order limit 10 ;

Order_ID,Order_Date,Profit,Customer_Name,Country,Category,Sub-Category
CA-2016-122581,21/8/2016,63.69,Jay Kimmel,United States,Furniture,Chairs
CA-2017-117485,23/9/2017,102.19,Bi1l Donatelli,United States,Technology,Accessories
US-2016-157490,6/10/2016,-14.92,Laurel Beltran,United States,Office Supplies,Binders
CA-2015-111703,2/7/2015,5.64,Karl Braun,United States,Office Supplies,Paper
CA-2014-108903,3/10/2014,-3.0,Denny Ordway,United States,Technology,Accessories
CA-2016-117583,27/11/2016,38.38,Cassandra Brandow,United States,Office Supplies,Binders
CA-2014-148488,10/12/2014,5.23,Sally Matthias,United States,Office Supplies,Paper
CA-2016-136434,1/12/2016,5.19,Rick Duston,United States,Furniture,Furnishings
CA-2014-160094,30/4/2014,214.0,Justin MacKendrick,United States,Office Supplies,Storage
CA-2017-141747,3/8/2017,4.18,Scot Coram,United States,Office Supplies,Storage


In [0]:
%sql
-- Profit By Year
SELECT
    YEAR(to_date(Order_Date, 'd/M/yyyy')) AS Year,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM final_enriched_order
GROUP BY YEAR(to_date(Order_Date, 'd/M/yyyy'))
ORDER BY Year;

-- Profit By Year and Product Category
SELECT
    YEAR(to_date(Order_Date, 'd/M/yyyy')) AS Year,
    Category,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM final_enriched_order
GROUP BY YEAR(to_date(Order_Date, 'd/M/yyyy')), Category
ORDER BY Year, Category;

-- Profit By Customer
SELECT
    Customer_Name,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM final_enriched_order
GROUP BY Customer_Name
ORDER BY Total_Profit DESC;

-- Profit By Customer + Year
SELECT
    Customer_Name,
    YEAR(to_date(Order_Date, 'd/M/yyyy')) AS Year,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM final_enriched_order
GROUP BY Customer_Name, YEAR(to_date(Order_Date, 'd/M/yyyy'))
ORDER BY Customer_Name, Year;

Customer_Name,Year,Total_Profit
null,2014,484.64
null,2015,2118.1
null,2016,288.42
null,2017,254.27
Dorothy Wardle,2014,6.63
Dorothy Wardle,2015,-371.25
Dorothy Wardle,2016,92.61
Dorothy Wardle,2017,27.5
=--Katharine Harms,2014,261.33
=--Katharine Harms,2016,43.9
